In [ ]:
"""
Presentation figures — aperiodic EEG / hormonal-contraceptive project.
ONE figure per file (single panel), sized for slides.

Colour palette (project brand):
    #FF66C4  bright pink   -> primary / Current-HC / positive
    #FFC7E9  light pink    -> secondary / Past-HC
    #B34789  deep magenta  -> accent / Never-user / negative
    #FFFFFF  white         -> background / diverging midpoint

Figures cover the variables analysed in the confound-controlled decode
(age, nicotine, alcohol, education, BDI, menstrual cycle phase), the aperiodic
parameters, and the FINAL model result.

Inputs
  <bids>/participants.tsv
  <phenotype>/bdi.tsv lifestyle.tsv hc_usage.tsv
  <bids>/derivatives/preproc/specparam/aperiodic_per_subject_channel.csv
Outputs (<bids>/derivatives/preproc/figures/)  -- each a standalone PNG
  01_sample_hc_groups.png        07_cycle_length_dist.png       13_exponent_vs_age.png
  02_age_dist.png                08_age_by_group.png            14_exponent_by_group.png
  03_education_dist.png          09_bdi_by_group.png            15_ec_vs_eo_exponent.png
  04_bdi_dist.png                10_cycle_phase_composition.png 16_correlation_heatmap.png
  05_alcohol_dist.png            11_cycle_phase_missingness.png 17_final_model.png
  06_nicotine_counts.png         12_confounds_correlation? (see 16)
  phenotype_merged.csv

Requires: pandas numpy matplotlib seaborn  (scikit-learn optional, for fig 17)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

# ============================================================
# CONFIG + STYLE
# ============================================================
BIDS_ROOT     = "/Users/elizabethkaplan/Desktop/ds007615/ds007615"   # nested root (matches classification.ipynb)
DERIV_ROOT    = os.path.join(BIDS_ROOT, "derivatives", "preproc")
PHENOTYPE_DIR = "/Users/elizabethkaplan/Desktop/phenotype"
PARTICIPANTS  = os.path.join(BIDS_ROOT, "participants.tsv")
APERIODIC_CSV = os.path.join(DERIV_ROOT, "specparam", "aperiodic_per_subject_channel.csv")
OUT_DIR       = os.path.join(DERIV_ROOT, "figures")
os.makedirs(OUT_DIR, exist_ok=True)

N_PERM_FINAL = 500   # permutation test for the final-model figure

# ---- brand palette ----
PINK  = "#FF66C4"; LIGHT = "#FFC7E9"; DARK = "#B34789"; WHITE = "#FFFFFF"
GROUP_PAL = {"Current HC": PINK, "Past HC": LIGHT, "Never": DARK}
CMAP_DIV = LinearSegmentedColormap.from_list("pink_div", [DARK, WHITE, PINK])
CMAP_SEQ = LinearSegmentedColormap.from_list("pink_seq", [WHITE, LIGHT, PINK, DARK])
def ramp(n): return [CMAP_SEQ(t) for t in np.linspace(0.30, 0.95, max(n, 1))]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160,
    "axes.titleweight": "bold", "axes.titlecolor": DARK,
    "axes.edgecolor": DARK, "axes.labelcolor": "#333333",
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#F3DEEA", "font.size": 13,
})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("saved", name)

def hist(col, label, name, unit=""):
    d = df[col].dropna()
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    sns.histplot(d, bins=14, kde=True, color=PINK, edgecolor=DARK, ax=ax)
    if ax.lines: ax.lines[0].set_color(DARK)
    ax.axvline(d.mean(), color=DARK, ls="--", lw=2)
    ax.set_title(label); ax.set_xlabel(unit or label); ax.set_ylabel("count")
    ax.text(0.97, 0.95, f"n = {d.notna().sum()}\nmean = {d.mean():.1f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=11, color=DARK)
    save(fig, name)

def violin_by_group(col, label, name, unit=""):
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    sns.violinplot(data=df, x="HC group", y=col, order=present_groups, hue="HC group",
                   palette=GROUP_PAL, legend=False, inner="quartile", cut=0, ax=ax)
    sns.stripplot(data=df, x="HC group", y=col, order=present_groups, color=DARK,
                  size=3.5, alpha=.55, ax=ax)
    ax.set_title(label); ax.set_xlabel(""); ax.set_ylabel(unit or label)
    save(fig, name)


# ============================================================
# LOAD + MERGE + DECODE
# ============================================================
def sid(path):
    d = pd.read_csv(path, sep="\t", na_values="n/a")
    d["subject"] = d["participant_id"].str.replace("sub-", "", regex=False)
    return d.set_index("subject")

parts = sid(PARTICIPANTS)
bdi   = sid(os.path.join(PHENOTYPE_DIR, "bdi.tsv"))
life  = sid(os.path.join(PHENOTYPE_DIR, "lifestyle.tsv"))
hc    = sid(os.path.join(PHENOTYPE_DIR, "hc_usage.tsv"))

df = parts.join([bdi.drop(columns="participant_id"),
                 life.drop(columns="participant_id"),
                 hc.drop(columns="participant_id")], how="left")

df["HC group"] = df["group"].map({1: "Current HC", 2: "Past HC", 3: "Never"})
df["Education"] = df["edu"].map({1: "Primary", 2: "Secondary (voc)",
                                 3: "Secondary (acad)", 4: "University"})
df["Menstrual phase"] = df["mens_phase"].map({1: "Follicular", 2: "Ovulatory", 3: "Luteal"})
df["Daily nicotine"] = df["daily_nicotine"].map({1: "Yes", 2: "No"})

if os.path.exists(APERIODIC_CSV):
    ap = pd.read_csv(APERIODIC_CSV, dtype={"subject": str})
    for acq in ["ec", "eo"]:
        sub = ap[ap["acq"] == acq].groupby("subject")[["exponent", "offset"]].mean()
        df[f"exponent_{acq}"] = sub["exponent"]; df[f"offset_{acq}"] = sub["offset"]
    HAS_AP = df["exponent_ec"].notna().any()
else:
    HAS_AP = False

df.to_csv(os.path.join(OUT_DIR, "phenotype_merged.csv"))
GROUP_ORDER = ["Current HC", "Past HC", "Never"]
present_groups = [g for g in GROUP_ORDER if g in df["HC group"].values]
PHASE_ORDER = [p for p in ["Follicular", "Ovulatory", "Luteal"] if p in df["Menstrual phase"].values]


# ============================================================
# 01 — sample composition (HC groups)
# ============================================================
fig, ax = plt.subplots(figsize=(7.2, 5.0))
sns.countplot(data=df, x="HC group", order=present_groups, hue="HC group",
              palette=GROUP_PAL, legend=False, ax=ax, edgecolor=DARK)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=12, color=DARK, fontweight="bold")
ax.set_title(f"Sample composition (N = {len(df)})"); ax.set_xlabel(""); ax.set_ylabel("participants")
save(fig, "01_sample_hc_groups.png")

# ============================================================
# 02–07 — distributions of variables we analysed
# ============================================================
hist("age", "Age distribution", "02_age_dist.png", "age (years)")

fig, ax = plt.subplots(figsize=(7.4, 5.0))
eo = [e for e in ["Primary", "Secondary (voc)", "Secondary (acad)", "University"]
      if e in df["Education"].values]
sns.countplot(data=df, y="Education", order=eo, hue="Education",
              palette=ramp(len(eo)), legend=False, ax=ax, edgecolor=DARK)
ax.set_title("Education"); ax.set_xlabel("participants"); ax.set_ylabel("")
save(fig, "03_education_dist.png")

hist("bdi_total", "Depression (BDI-II total)", "04_bdi_dist.png", "BDI-II total")
hist("alcohol_units", "Alcohol use", "05_alcohol_dist.png", "units per week")

fig, ax = plt.subplots(figsize=(6.2, 5.0))
d = df["Daily nicotine"].dropna()
sns.countplot(x=d, order=["No", "Yes"], hue=d, palette={"No": LIGHT, "Yes": PINK},
              legend=False, ax=ax, edgecolor=DARK)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=12, color=DARK, fontweight="bold")
ax.set_title("Daily nicotine use"); ax.set_xlabel(""); ax.set_ylabel("participants")
save(fig, "06_nicotine_counts.png")

hist("cycle_length", "Menstrual cycle length", "07_cycle_length_dist.png", "days")

# ============================================================
# 08–09 — confound balance across HC groups
# ============================================================
violin_by_group("age", "Age by HC status", "08_age_by_group.png", "age (years)")
violin_by_group("bdi_total", "Depression (BDI-II) by HC status", "09_bdi_by_group.png", "BDI-II total")

# ============================================================
# 10 — cycle phase composition by HC group
# ============================================================
ph = df.dropna(subset=["Menstrual phase"])
ct = pd.crosstab(ph["HC group"], ph["Menstrual phase"]).reindex(present_groups)[PHASE_ORDER]
fig, ax = plt.subplots(figsize=(7.4, 5.2))
ct.plot(kind="bar", stacked=True, color=ramp(len(PHASE_ORDER)), edgecolor="white", ax=ax)
ax.set_title("Cycle phase at recording\n(subjects with data)")
ax.set_xlabel(""); ax.set_ylabel("participants"); ax.tick_params(axis="x", rotation=0)
ax.legend(title="", fontsize=11)
save(fig, "10_cycle_phase_composition.png")

# ============================================================
# 11 — cycle-phase data availability (the missingness caveat)
# ============================================================
df["Phase data"] = np.where(df["mens_phase"].notna(), "Reported", "Missing")
miss = (pd.crosstab(df["HC group"], df["Phase data"], normalize="index")
        .reindex(present_groups)[["Reported", "Missing"]])
fig, ax = plt.subplots(figsize=(7.4, 5.4))
miss.plot(kind="bar", stacked=True, color=[PINK, DARK], edgecolor="white", ax=ax)
ax.set_title("Cycle-phase data availability")
ax.set_xlabel(""); ax.set_ylabel("fraction of group"); ax.tick_params(axis="x", rotation=0)
ax.legend(title="", fontsize=11)
ax.text(0.5, -0.20, "Missingness tracks HC use → indicator alone decodes at AUC ≈ 0.66.\n"
                    "Phase is therefore mean-imputed; no missingness indicator is used as a confound.",
        transform=ax.transAxes, ha="center", va="top", fontsize=10.5, color=DARK)
save(fig, "11_cycle_phase_missingness.png")

# ============================================================
# 13–15 — aperiodic parameters
# ============================================================
if HAS_AP:
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    sns.regplot(data=df, x="age", y="exponent_ec", color=PINK,
                scatter_kws=dict(s=55, alpha=.75, edgecolor=DARK), line_kws=dict(color=DARK), ax=ax)
    r = df[["age", "exponent_ec"]].corr().iloc[0, 1]
    ax.set_title("Aperiodic exponent vs age (eyes-closed)")
    ax.set_xlabel("age (years)"); ax.set_ylabel("exponent")
    ax.text(0.03, 0.05, f"r = {r:.2f}", transform=ax.transAxes, fontsize=12, color=DARK, fontweight="bold")
    save(fig, "13_exponent_vs_age.png")

    violin_by_group("exponent_ec", "Aperiodic exponent by HC status (EC)", "14_exponent_by_group.png", "exponent")

    fig, ax = plt.subplots(figsize=(7.0, 6.2))
    lim = [min(df["exponent_ec"].min(), df["exponent_eo"].min()),
           max(df["exponent_ec"].max(), df["exponent_eo"].max())]
    sns.scatterplot(data=df, x="exponent_ec", y="exponent_eo", hue="HC group",
                    hue_order=present_groups, palette=GROUP_PAL, s=70, edgecolor=DARK, ax=ax)
    ax.plot(lim, lim, "--", color=DARK, lw=1.2)
    ax.set_title("Eyes-closed vs eyes-open exponent")
    ax.set_xlabel("eyes-closed exponent"); ax.set_ylabel("eyes-open exponent")
    ax.legend(title="", fontsize=10)
    save(fig, "15_ec_vs_eo_exponent.png")

# ============================================================
# 16 — correlation heatmap (analysis-relevant continuous vars)
# ============================================================
corr_vars = [c for c in ["age", "bdi_total", "alcohol_units", "cycle_length",
                         "exponent_ec", "offset_ec", "exponent_eo", "offset_eo"] if c in df.columns]
corr = df[corr_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
nice = {"age": "age", "bdi_total": "BDI", "alcohol_units": "alcohol", "cycle_length": "cycle len",
        "exponent_ec": "exp EC", "offset_ec": "off EC", "exponent_eo": "exp EO", "offset_eo": "off EO"}
labels = [nice.get(c, c) for c in corr_vars]
fig, ax = plt.subplots(figsize=(8.5, 7.2))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, linecolor="white", xticklabels=labels, yticklabels=labels,
            cbar_kws={"shrink": .7, "label": "Pearson r"}, annot_kws={"size": 10}, ax=ax)
ax.set_title("Correlations among analysed measures")
save(fig, "16_correlation_heatmap.png")

# ============================================================
# 17 — FINAL MODEL: HC decode after full confound control
#      complete-case sample (cycle phase observed); three bars.
# ============================================================
try:
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import LeaveOneOut, cross_val_predict
    from sklearn.metrics import roc_auc_score

    class ConfoundRegressor(BaseEstimator, TransformerMixin):
        def __init__(self, n_confounds): self.n_confounds = n_confounds
        def fit(self, X, y=None):
            Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
            self.c_mean_ = C.mean(0)
            d = np.column_stack([np.ones(len(C)), C - self.c_mean_])
            self.beta_ = np.linalg.lstsq(d, Xs, rcond=None)[0]; return self
        def transform(self, X):
            Xs, C = X[:, :-self.n_confounds], X[:, -self.n_confounds:]
            d = np.column_stack([np.ones(len(C)), C - self.c_mean_])
            return Xs - d @ self.beta_

    lg = lambda: LogisticRegression(penalty="l2", C=0.1, max_iter=2000, solver="liblinear")
    basep  = lambda: Pipeline([("i", SimpleImputer()), ("s", StandardScaler()), ("l", lg())])
    residp = lambda nc: Pipeline([("i", SimpleImputer()), ("c", ConfoundRegressor(nc)),
                                  ("s", StandardScaler()), ("l", lg())])
    def auc(pipe, X, yy):
        p = cross_val_predict(pipe, X, yy, cv=LeaveOneOut(), method="predict_proba")[:, 1]
        return roc_auc_score(yy, p)

    apw = pd.read_csv(APERIODIC_CSV, dtype={"subject": str})
    apw = apw[apw["acq"].isin(["ec", "eo"])]
    wide = apw.pivot_table(index="subject", columns=["acq", "channel"], values=["exponent", "offset"])
    wide.columns = [f"{p}_{a}_{c}" for (p, a, c) in wide.columns]; wide = wide.sort_index()
    S = wide.index
    yv  = (parts.loc[S, "group"] == 1).astype(int).values
    Xe  = wide.values
    age_ = parts.loc[S, "age"].astype(float).values
    nic_ = (life.loc[S, "daily_nicotine"] == 1).astype(float).values
    alc_ = life.loc[S, "alcohol_units"].astype(float).values
    edu_ = parts.loc[S, "edu"].astype(float).values
    bdi_ = bdi.loc[S, "bdi_total"].astype(float).values
    mph  = hc.loc[S, "mens_phase"].astype(float).values
    hp   = ~np.isnan(mph)                       # complete-case: observed cycle phase

    Z = np.column_stack([age_[hp], nic_[hp], alc_[hp], edu_[hp], bdi_[hp],
                         (mph[hp] == 2).astype(float), (mph[hp] == 3).astype(float)])
    X, yy = Xe[hp], yv[hp]; nc = Z.shape[1]
    a_cov = auc(basep(), Z, yy)
    a_eeg = auc(basep(), X, yy)
    a_res = auc(residp(nc), np.hstack([X, Z]), yy)
    # permutation p on the residualized model
    rng = np.random.RandomState(42); null = np.empty(N_PERM_FINAL); Xrz = np.hstack([X, Z])
    for i in range(N_PERM_FINAL):
        yp = rng.permutation(yy)
        pr = cross_val_predict(residp(nc), Xrz, yp, cv=LeaveOneOut(),
                               method="predict_proba", n_jobs=-1)[:, 1]
        null[i] = roc_auc_score(yp, pr)
    p_res = (1 + np.sum(null >= a_res)) / (N_PERM_FINAL + 1)

    fig, ax = plt.subplots(figsize=(8.0, 6.0))
    bars = ["Confounds\nonly", "EEG\nonly", "EEG | confounds\n(residualized)"]
    vals = [a_cov, a_eeg, a_res]
    cols = [LIGHT, PINK, DARK]
    b = ax.bar(bars, vals, color=cols, edgecolor=DARK, width=0.62)
    ax.axhline(0.5, color="0.5", ls="--", lw=1.6, zorder=0)
    ax.text(-0.45, 0.505, "chance", color="0.4", fontsize=10, ha="left", va="bottom")
    for rect, v in zip(b, vals):
        ax.annotate(f"{v:.2f}", (rect.get_x() + rect.get_width()/2, v),
                    ha="center", va="bottom", fontsize=14, color=DARK, fontweight="bold")
    ax.annotate(f"permutation p = {p_res:.3f}",
                (2, a_res), xytext=(2, a_res + 0.06), ha="center", fontsize=12,
                color=DARK, fontweight="bold")
    ax.set_ylim(0.4, 0.82); ax.set_ylabel("LOSO AUC")
    ax.set_title(f"HC use decodes from aperiodic EEG\nafter full confound control  (n = {int(hp.sum())})")
    ax.text(0.5, -0.16, "Confounds = age · nicotine · alcohol · education · BDI · cycle phase",
            transform=ax.transAxes, ha="center", va="top", fontsize=10.5, color="0.4")
    save(fig, "17_final_model.png")
except Exception as e:
    print(f"fig 17 skipped ({type(e).__name__}: {e})")

print(f"\nAll figures -> {OUT_DIR}")
